In [ ]:
import time, uuid
_GUARD_STARTED = time.monotonic()
import json, os
os.environ['SF_COMMIT_TASKS'] = '8'
os.environ['SF_DEADLINE_S'] = '41400'
os.environ['SF_METHOD'] = 'score'
os.environ['SF_MODE'] = 'submit'
os.environ['SF_N_GEN'] = '8'
os.environ['SF_N_SCORE'] = '8'
os.environ['SF_TASK_CAP_S'] = '900'
os.environ['SF_TTT_ALPHA'] = '64'
os.environ['SF_TTT_FRAC'] = '0.7'
os.environ['SF_TTT_LR'] = '1e-4'
os.environ['SF_TTT_MAX_TOKENS'] = '8192'
os.environ['SF_TTT_RANK'] = '64'
os.environ['SF_TTT_STEPS'] = '48'
CODE_DIR = os.environ.get('SF_KAGGLE_ROOT', '/kaggle') + '/working/sf_code_guarded_' + uuid.uuid4().hex
CODE_SHA = '2d876c820e923316cd9f8f3971c3a4a86a22dff8ba627056f0b8a01fb668eb24'
FILES = json.loads('{"sf/__init__.py": "", "sf/grids.py": "\\"\\"\\"Grid utilities: validation, the eight symmetries of the square (D8), colour permutations, and task augmentation\\nwith exact inverses. Written for this project from first principles.\\"\\"\\"\\nimport random\\n\\nimport numpy as np\\n\\nN_COLOURS = 10\\nMAX_SIDE = 30\\n\\n\\ndef as_array(grid):\\n    a = np.asarray(grid, dtype=np.int64)\\n    if a.ndim != 2:\\n        raise ValueError(\\"grid must be 2-D\\")\\n    return a\\n\\n\\ndef is_valid_grid(grid):\\n    \\"\\"\\"A valid ARC grid: a non-empty rectangular list of rows of ints in 0..9, at most 30 x 30.\\"\\"\\"\\n    if not isinstance(grid, (list, tuple)) or not grid:\\n        return False\\n    w = None\\n    for row in grid:\\n        if not isinstance(row, (list, tuple)) or not row:\\n            return False\\n        if w is None:\\n            w = len(row)\\n        elif len(row) != w:\\n            return False\\n        for v in row:\\n            if isinstance(v, bool) or not isinstance(v, (int, np.integer)) or not 0 <= int(v) < N_COLOURS:\\n                return False\\n    return len(grid) <= MAX_SIDE and w <= MAX_SIDE\\n\\n\\ndef to_list(a):\\n    return [[int(v) for v in row] for row in np.asarray(a)]\\n\\n\\ndef key(grid):\\n    \\"\\"\\"Hashable canonical key of a grid.\\"\\"\\"\\n    a = np.asarray(grid)\\n    return (a.shape[0], a.shape[1], a.astype(np.int8).tobytes())\\n\\n\\n# D8: t in 0..7; k = t % 4 quarter turns counter-clockwise, then a left-right flip when t >= 4.\\ndef d8(a, t):\\n    a = np.rot90(np.asarray(a), t % 4)\\n    if t >= 4:\\n        a = np.fliplr(a)\\n    return np.ascontiguousarray(a)\\n\\n\\ndef d8_inv(a, t):\\n    a = np.asarray(a)\\n    if t >= 4:\\n        a = np.fliplr(a)\\n    return np.ascontiguousarray(np.rot90(a, -(t % 4)))\\n\\n\\ndef identity_perm():\\n    return np.arange(N_COLOURS)\\n\\n\\ndef random_perm(rng, keep_zero=True):\\n    p = np.arange(N_COLOURS)\\n    if keep_zero:\\n        rest = list(range(1, N_COLOURS))\\n        rng.shuffle(rest)\\n        p[1:] = rest\\n    else:\\n        rest = list(range(N_COLOURS))\\n        rng.shuffle(rest)\\n        p[:] = rest\\n    return p\\n\\n\\nclass Aug:\\n    \\"\\"\\"One augmentation: a D8 element, a colour permutation (applied after the symmetry) and a demonstration order.\\"\\"\\"\\n\\n    def __init__(self, t=0, perm=None, order=None):\\n        self.t = int(t)\\n        self.perm = identity_perm() if perm is None else np.asarray(perm, dtype=np.int64)\\n        self.inv_perm = np.argsort(self.perm)\\n        self.order = None if order is None else list(order)\\n\\n    def fwd(self, g):\\n        return self.perm[d8(as_array(g), self.t)]\\n\\n    def inv(self, g):\\n        return d8_inv(self.inv_perm[as_array(g)], self.t)\\n\\n    def apply_task(self, train_pairs, test_inputs):\\n        pairs = [(self.fwd(i), self.fwd(o)) for i, o in train_pairs]\\n        if self.order is not None:\\n            pairs = [pairs[j] for j in self.order]\\n        return pairs, [self.fwd(g) for g in test_inputs]\\n\\n    def describe(self):\\n        return {\\"t\\": self.t, \\"perm\\": [int(v) for v in self.perm], \\"order\\": self.order}\\n\\n    @staticmethod\\n    def from_desc(d):\\n        return Aug(d[\\"t\\"], d[\\"perm\\"], d.get(\\"order\\"))\\n\\n\\ndef make_augs(n, n_train, seed, keep_zero=True, shuffle_order=True):\\n    \\"\\"\\"n augmentations: the identity first, then the other seven symmetries with identity colours, then random\\n    (symmetry, colour permutation, order) triples. Deterministic in seed.\\"\\"\\"\\n    rng = random.Random(seed)\\n    augs = [Aug(0)]\\n    for t in range(1, 8):\\n        if len(augs) >= n:\\n            break\\n        augs.append(Aug(t))\\n    while len(augs) < n:\\n        t = rng.randrange(8)\\n        perm = random_perm(rng, keep_zero)\\n        order = list(range(n_train))\\n        if shuffle_order:\\n            rng.shuffle(order)\\n        augs.append(Aug(t, perm, order))\\n    return augs[:n]\\n\\n\\ndef palette(g):\\n    return set(int(v) for v in np.unique(as_array(g)))\\n", "sf/lm.py": "\\"\\"\\"The grid language model: loading, prompt prefill, batched constrained greedy decoding and teacher-forced scoring.\\n\\nOwn implementation on transformers\' public model API (Apache-2.0). Prompts are prefilled one at a time (no padding,\\nso the causal attention kernels apply), then their key/value caches are left-padded into one batch for decoding.\\n\\"\\"\\"\\nimport time\\n\\nimport torch\\nfrom transformers import AutoModelForCausalLM, DynamicCache, StaticCache\\n\\nfrom . import textfmt as F\\n\\nPREFILL_CHUNK = 4096                 # tokens per prefill forward (1024 on pre-Ampere GPUs, see LM)\\n\\n\\ndef make_cache(kv):\\n    \\"\\"\\"A DynamicCache holding the given per-layer (keys, values).\\"\\"\\"\\n    try:\\n        return DynamicCache(ddp_cache_data=kv)\\n    except TypeError:\\n        cache = DynamicCache()\\n        for i, (k, v) in enumerate(kv):\\n            cache.update(k, v, i)\\n        return cache\\n\\n\\nclass LM:\\n    def __init__(self, path, device=\\"cuda:0\\", dtype=None, attn=\\"sdpa\\"):\\n        self.device = torch.device(device)\\n        if dtype is None:\\n            if self.device.type != \\"cuda\\":\\n                dtype = torch.float32\\n            else:\\n                major = torch.cuda.get_device_capability(self.device)[0]\\n                dtype = torch.bfloat16 if major >= 8 else torch.float16\\n        self.dtype = dtype\\n        self.chunk = PREFILL_CHUNK\\n        self.use_static = True\\n        if self.device.type == \\"cuda\\" and torch.cuda.get_device_capability(self.device)[0] < 8:\\n            self.chunk = 1024\\n        t0 = time.time()\\n        self.model = AutoModelForCausalLM.from_pretrained(path, dtype=dtype, device_map={\\"\\": str(self.device)},\\n                                                          attn_implementation=attn)\\n        self.model.eval()\\n        self.load_s = time.time() - t0\\n        self.stats = {\\"prefill_tokens\\": 0, \\"decode_steps\\": 0, \\"decode_seq_steps\\": 0, \\"score_tokens\\": 0,\\n                      \\"prefill_s\\": 0.0, \\"decode_s\\": 0.0, \\"score_s\\": 0.0}\\n\\n    # ----- prefill -------------------------------------------------------------------------------------------\\n    @torch.no_grad()\\n    def prefill(self, ids):\\n        \\"\\"\\"Run a prompt (batch 1); return (cache, log-probs of the next token).\\"\\"\\"\\n        t0 = time.time()\\n        cache = DynamicCache()\\n        out = None\\n        for s in range(0, len(ids), self.chunk):\\n            chunk = torch.tensor([ids[s:s + self.chunk]], device=self.device)\\n            out = self.model(input_ids=chunk, past_key_values=cache, use_cache=True, logits_to_keep=1)\\n            cache = out.past_key_values\\n        lp = torch.log_softmax(out.logits[0, -1].float(), -1)\\n        self.stats[\\"prefill_tokens\\"] += len(ids)\\n        self.stats[\\"prefill_s\\"] += time.time() - t0\\n        return cache, lp\\n\\n    # ----- constrained greedy decoding -----------------------------------------------------------------------\\n    @torch.no_grad()\\n    def _static_batch(self, prompts, max_len):\\n        \\"\\"\\"Prefill each prompt alone and copy its keys/values, left-padded, into one preallocated static cache\\n        (decoding then writes in place instead of re-concatenating the whole cache at every step).\\"\\"\\"\\n        cfg = self.model.config\\n        B, lens = len(prompts), [len(p) for p in prompts]\\n        L = max(lens)\\n        cache = StaticCache(config=cfg, max_cache_len=max_len)\\n        cache.early_initialization(B, cfg.num_key_value_heads, cfg.head_dim, self.dtype, self.device)\\n        first = []\\n        for b, p in enumerate(prompts):\\n            c, lp = self.prefill(p)\\n            for dst, src in zip(cache.layers, c.layers):\\n                dst.keys[b, :, L - lens[b]:L].copy_(src.keys[0])\\n                dst.values[b, :, L - lens[b]:L].copy_(src.values[0])\\n            first.append(lp)\\n            del c\\n        for layer in cache.layers:\\n            if torch.is_tensor(layer.cumulative_length):\\n                layer.cumulative_length.fill_(L)\\n            else:\\n                layer.cumulative_length = L\\n        return cache, torch.stack(first), lens, L\\n\\n    @torch.no_grad()\\n    def _dynamic_batch(self, prompts):\\n        \\"\\"\\"Fallback when the static cache API is unavailable: left-padded concatenation into a DynamicCache.\\"\\"\\"\\n        caches, first = [], []\\n        for p in prompts:\\n            c, lp = self.prefill(p)\\n            caches.append(c)\\n            first.append(lp)\\n        lens = [len(p) for p in prompts]\\n        L = max(lens)\\n        merged = []\\n        for li in range(len(caches[0].layers)):\\n            ks, vs = [], []\\n            for b, c in enumerate(caches):\\n                pad = L - lens[b]\\n                ks.append(torch.nn.functional.pad(c.layers[li].keys, (0, 0, pad, 0)))\\n                vs.append(torch.nn.functional.pad(c.layers[li].values, (0, 0, pad, 0)))\\n            merged.append((torch.cat(ks, 0), torch.cat(vs, 0)))\\n        del caches\\n        return make_cache(merged), torch.stack(first), lens, L\\n\\n    @torch.no_grad()\\n    def greedy(self, prompts, max_new=F.MAX_OUT_TOKENS):\\n        \\"\\"\\"Constrained greedy decoding of a batch of prompts. Returns, per prompt, (tokens incl. end, sum logp).\\"\\"\\"\\n        B = len(prompts)\\n        L0 = max(len(p) for p in prompts)\\n        static = self.use_static\\n        if static:\\n            try:\\n                cache, lp, lens, L = self._static_batch(prompts, L0 + max_new)\\n            except (TypeError, AttributeError, NotImplementedError):\\n                self.use_static = static = False\\n        if not static:\\n            cache, lp, lens, L = self._dynamic_batch(prompts)\\n        t0 = time.time()\\n        mask = torch.zeros((B, L + max_new), dtype=torch.long, device=self.device)\\n        for b in range(B):\\n            mask[b, L - lens[b]:L] = 1\\n        cons = [F.GridConstraint() for _ in range(B)]\\n        toks = [[] for _ in range(B)]\\n        score = [0.0] * B\\n        pos = torch.tensor(lens, device=self.device)\\n        V = lp.shape[-1]\\n        for step in range(max_new):\\n            allow = torch.zeros((B, V), dtype=torch.bool)\\n            live = []\\n            for b in range(B):\\n                a = cons[b].allowed()\\n                if a:\\n                    allow[b, list(a)] = True\\n                    live.append(b)\\n            if not live:\\n                break\\n            allow = allow.to(self.device, non_blocking=True)\\n            nxt = lp.masked_fill(~allow, float(\\"-inf\\")).argmax(-1)\\n            chosen = lp.gather(1, nxt[:, None])[:, 0]\\n            both = torch.stack([nxt.float(), chosen]).tolist()\\n            nl = [int(v) for v in both[0]]\\n            for b in live:\\n                toks[b].append(nl[b])\\n                score[b] += both[1][b]\\n                cons[b].step(nl[b])\\n            if all(c.done for c in cons):\\n                break\\n            mask[:, L + step] = 1\\n            if static:\\n                out = self.model(input_ids=nxt[:, None], attention_mask=mask, position_ids=pos[:, None],\\n                                 past_key_values=cache, use_cache=True,\\n                                 cache_position=torch.tensor([L + step], device=self.device))\\n            else:\\n                out = self.model(input_ids=nxt[:, None], attention_mask=mask[:, :L + step + 1],\\n                                 position_ids=pos[:, None], past_key_values=cache, use_cache=True)\\n                cache = out.past_key_values\\n            pos = pos + 1\\n            lp = torch.log_softmax(out.logits[:, -1].float(), -1)\\n            self.stats[\\"decode_steps\\"] += 1\\n            self.stats[\\"decode_seq_steps\\"] += len(live)\\n        del cache\\n        self.stats[\\"decode_s\\"] += time.time() - t0\\n        return [(toks[b], score[b]) for b in range(B)]\\n\\n    # ----- teacher-forced scoring ----------------------------------------------------------------------------\\n    @torch.no_grad()\\n    def score(self, prompt, targets, cache=None, first_lp=None):\\n        \\"\\"\\"Sum of log-probs of each target token sequence after the prompt (prompt cache reused, cropped back).\\"\\"\\"\\n        if cache is None:\\n            cache, first_lp = self.prefill(prompt)\\n        t0 = time.time()\\n        L = len(prompt)\\n        res = []\\n        for tg in targets:\\n            total = float(first_lp[tg[0]])\\n            if len(tg) > 1:\\n                x = torch.tensor([tg[:-1]], device=self.device)\\n                out = self.model(input_ids=x, past_key_values=cache, use_cache=True,\\n                                 position_ids=torch.arange(L, L + len(tg) - 1, device=self.device)[None])\\n                lps = torch.log_softmax(out.logits[0].float(), -1)\\n                total += float(lps.gather(1, torch.tensor(tg[1:], device=self.device)[:, None]).sum())\\n                cache = out.past_key_values\\n                cache.crop(-(len(tg) - 1))          # drop the target tokens again\\n            self.stats[\\"score_tokens\\"] += len(tg)\\n            res.append(total)\\n        self.stats[\\"score_s\\"] += time.time() - t0\\n        return res\\n", "sf/pilot.py": "\\"\\"\\"TTT pilot (authorized by the author on 26 Oct; Kaggle L4x4 only, at most 2 h of runtime in total).\\n\\n  smoke: one LoRA configuration per GPU. Peak memory and time per step at the longest pilot sequence and at 8,192\\n         tokens, adapter inference time, and a leave-one-demonstration-out (LOO) check on three NON-pilot puzzles\\n         (adapt on all demonstrations but the last, then predict the last one\'s output: its NLL and greedy vote\\n         before and after). Demonstrations only.\\n  pilot: for each pilot puzzle, in one process: the LOO check, the no-TTT arm (fresh adapter = the base model) and\\n         the TTT arm (a fresh adapter trained on all demonstrations), both with the same inference. Query answers\\n         are never read.\\n  python -m sf.pilot smoke|pilot --rank R --challenges C --model M --out DIR --tasks a,b,c [--cfg JSON]\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport time\\n\\nimport numpy as np\\nimport torch\\n\\nfrom . import grids as G\\nfrom . import textfmt as F\\nfrom .lm import LM\\nfrom .predict import predict_output, task_seed\\nfrom .runner import atomic_json\\nfrom .ttt import TTT\\n\\n\\ndef peak_reset():\\n    if torch.cuda.is_available():\\n        torch.cuda.synchronize()\\n        torch.cuda.reset_peak_memory_stats()\\n\\n\\ndef peak():\\n    if not torch.cuda.is_available():\\n        return 0.0\\n    torch.cuda.synchronize()\\n    return round(torch.cuda.max_memory_allocated() / 1e9, 2)\\n\\n\\ndef device():\\n    return os.environ.get(\\"SF_DEVICE\\", \\"cuda:0\\")\\n\\n\\ndef pairs_of(task):\\n    return [(np.asarray(p[\\"input\\"]), np.asarray(p[\\"output\\"])) for p in task[\\"train\\"]]\\n\\n\\ndef loo_eval(lm, pairs, n_aug, seed):\\n    \\"\\"\\"Predict the last demonstration\'s output from the others: NLL of the true output (identity prompt), and\\n    whether the top vote of a greedy decode under n_aug augmentations equals it.\\"\\"\\"\\n    ctx, (hin, hout) = pairs[:-1], pairs[-1]\\n    p = F.prompt_ids(ctx, hin)\\n    nll = -lm.score(p, [F.target_ids(hout)])[0]\\n    res = predict_output(lm, ctx, hin, n_gen=n_aug, n_score=0, seed=seed)\\n    pool = sorted(res[\\"candidates\\"], key=lambda c: -c[\\"votes\\"])\\n    top = pool[0][\\"grid\\"] if pool else None\\n    hit_any = any(c[\\"grid\\"] == G.to_list(hout) for c in pool)\\n    return {\\"nll\\": round(nll, 4), \\"top_vote_correct\\": top == G.to_list(hout), \\"in_pool\\": hit_any,\\n            \\"n_cands\\": len(pool), \\"s\\": res[\\"total_s\\"]}\\n\\n\\ndef synthetic_pairs(n_tokens, rng):\\n    \\"\\"\\"Random 30 x 30 pairs until the LOO sequence reaches about n_tokens (memory test only).\\"\\"\\"\\n    pairs = []\\n    while sum(len(F.grid_ids(i)) + len(F.grid_ids(o)) + 12 for i, o in pairs) < n_tokens:\\n        pairs.append((rng.integers(0, 10, (30, 30)), rng.integers(0, 10, (30, 30))))\\n    return pairs\\n\\n\\ndef smoke(a, cfg, ch):\\n    lm = LM(a.model, device=device())\\n    out = {\\"cfg\\": cfg, \\"load_s\\": round(lm.load_s, 1),\\n           \\"gpu\\": torch.cuda.get_device_name(0) if torch.cuda.is_available() else \\"cpu\\"}\\n    peak_reset()\\n    ttt = TTT(lm, rank=cfg[\\"rank\\"], alpha=cfg[\\"alpha\\"], lr=cfg[\\"lr\\"], max_tokens=cfg.get(\\"max_tokens\\", 8192))\\n    out[\\"lora_params\\"] = ttt.n_params\\n    out[\\"mem_after_wrap_gb\\"] = peak()\\n    big = cfg[\\"big_task\\"]\\n    rng = np.random.default_rng(0)\\n    for name, pairs in ((\\"longest_pilot\\", pairs_of(ch[big])), (\\"synthetic_8192\\", synthetic_pairs(8000, rng))):\\n        peak_reset()\\n        st = ttt.train(pairs, steps=cfg.get(\\"mem_steps\\", 4), seed=1, log_every=1)\\n        out[f\\"mem_{name}\\"] = {\\"peak_gb\\": peak(), \\"s_per_step\\": round(st[\\"train_s\\"] / max(1, st[\\"steps\\"]), 2),\\n                              \\"tokens_per_step\\": st[\\"tokens\\"] // max(1, st[\\"steps\\"]), \\"loss\\": st[\\"loss_curve\\"]}\\n        atomic_json(os.path.join(a.out, f\\"smoke_{a.rank}.json\\"), out)\\n    peak_reset()\\n    ttt.reset(seed=3)\\n    t0 = time.time()\\n    x = ch[big][\\"test\\"][0][\\"input\\"]\\n    r = predict_output(lm, pairs_of(ch[big]), np.asarray(x), cfg[\\"n_gen\\"], cfg[\\"n_score\\"], task_seed(big, 0))\\n    out[\\"adapter_inference\\"] = {\\"task\\": big, \\"s\\": round(time.time() - t0, 2), \\"peak_gb\\": peak(),\\n                                \\"n_cands\\": len(r[\\"candidates\\"]), \\"prompt_tokens\\": r[\\"prompt_tokens\\"]}\\n    out[\\"loo\\"] = []\\n    for tid in cfg[\\"loo_tasks\\"]:\\n        pairs = pairs_of(ch[tid])\\n        ttt.reset(seed=task_seed(tid, 99))\\n        before = loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7))\\n        peak_reset()\\n        st = ttt.train(pairs[:-1], steps=cfg[\\"steps\\"], seed=task_seed(tid, 99))\\n        after = loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7))\\n        out[\\"loo\\"].append({\\"task\\": tid, \\"before\\": before, \\"after\\": after, \\"train\\": st, \\"peak_gb\\": peak()})\\n        atomic_json(os.path.join(a.out, f\\"smoke_{a.rank}.json\\"), out)\\n    out[\\"lm_stats\\"] = lm.stats\\n    out[\\"done\\"] = True\\n    atomic_json(os.path.join(a.out, f\\"smoke_{a.rank}.json\\"), out)\\n\\n\\ndef pilot(a, cfg, ch):\\n    lm = LM(a.model, device=device())\\n    ttt = TTT(lm, rank=cfg[\\"rank\\"], alpha=cfg[\\"alpha\\"], lr=cfg[\\"lr\\"], max_tokens=cfg.get(\\"max_tokens\\", 8192))\\n    for tid in a.tasks.split(\\",\\"):\\n        path = os.path.join(a.out, \\"pilot\\", f\\"{tid}.json\\")\\n        if os.path.exists(path):\\n            continue\\n        pairs = pairs_of(ch[tid])\\n        rec = {\\"task\\": tid, \\"cfg\\": cfg, \\"n_demos\\": len(pairs)}\\n        seed = task_seed(tid, 99)\\n        if cfg.get(\\"loo\\", True) and len(pairs) >= 2:\\n            ttt.reset(seed)\\n            before = loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7))\\n            peak_reset()\\n            st = ttt.train(pairs[:-1], steps=cfg[\\"steps\\"], seed=seed)\\n            rec[\\"loo\\"] = {\\"before\\": before, \\"after\\": loo_eval(lm, pairs, cfg[\\"loo_augs\\"], task_seed(tid, 7)),\\n                          \\"train\\": st, \\"peak_gb\\": peak()}\\n        for arm in (\\"no_ttt\\", \\"ttt\\"):\\n            ttt.reset(seed)\\n            peak_reset()\\n            st = None\\n            if arm == \\"ttt\\":\\n                st = ttt.train(pairs, steps=cfg[\\"steps\\"], seed=seed)\\n            t0 = time.time()\\n            outs = [predict_output(lm, pairs, np.asarray(x[\\"input\\"]), cfg[\\"n_gen\\"], cfg[\\"n_score\\"],\\n                                   task_seed(tid, j)) for j, x in enumerate(ch[tid][\\"test\\"])]\\n            rec[arm] = {\\"outputs\\": outs, \\"train\\": st, \\"infer_s\\": round(time.time() - t0, 2), \\"peak_gb\\": peak()}\\n        atomic_json(path, rec)\\n\\n\\ndef main():\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"mode\\", choices=[\\"smoke\\", \\"pilot\\"])\\n    ap.add_argument(\\"--rank\\", type=int, default=0)\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--model\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--tasks\\", default=\\"\\")\\n    ap.add_argument(\\"--cfg\\", required=True)\\n    a = ap.parse_args()\\n    cfg = json.loads(a.cfg)\\n    ch = json.load(open(a.challenges))\\n    os.makedirs(os.path.join(a.out, \\"pilot\\"), exist_ok=True)\\n    (smoke if a.mode == \\"smoke\\" else pilot)(a, cfg, ch)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "sf/predict.py": "\\"\\"\\"Per-task candidate generation without test-time training.\\n\\nFor each test input: build the prompt under several augmentations (D8 symmetry, colour permutation keeping 0,\\ndemonstration order), decode greedily with the grid constraint, map every decoded grid back to the original frame,\\nand score every distinct candidate under every augmentation (teacher forcing). The full candidate pool is returned\\nso that selection rules can be compared offline on the same pool.\\n\\"\\"\\"\\nimport copy\\nimport math\\nimport time\\nimport zlib\\n\\nimport numpy as np\\n\\nfrom . import grids as G\\nfrom . import textfmt as F\\n\\nMAX_PROMPT = 12000            # tokens; longer prompts drop their longest demonstrations first\\nBATCH_TOKENS = 64000          # cap on (batch size x (prompt + answer)) tokens for one decoding batch\\n\\n\\ndef task_seed(task_id, idx):\\n    return zlib.crc32(f\\"{task_id}:{idx}\\".encode()) & 0x7FFFFFFF\\n\\n\\ndef fit_prompt(pairs, test_input, max_tokens=MAX_PROMPT):\\n    \\"\\"\\"Drop the longest demonstrations until the prompt fits; returns (kept pair indices, n_tokens).\\"\\"\\"\\n    keep = list(range(len(pairs)))\\n\\n    def n_tok(ix):\\n        return len(F.prompt_ids([pairs[i] for i in ix], test_input))\\n    n = n_tok(keep)\\n    while n > max_tokens and len(keep) > 1:\\n        longest = max(keep, key=lambda i: len(F.grid_ids(pairs[i][0])) + len(F.grid_ids(pairs[i][1])))\\n        keep.remove(longest)\\n        n = n_tok(keep)\\n    return keep, n\\n\\n\\ndef predict_output(lm, pairs, test_input, n_gen, n_score, seed, max_prompt=MAX_PROMPT, deadline=None, _prefix_token=None):\\n    \\"\\"\\"Candidate pool for one test input. n_gen augmentations decode; n_score >= n_gen augmentations score.\\n    The first n_gen scoring augmentations are the decoding ones.\\"\\"\\"\\n    t0 = time.time()\\n    keep, n_prompt = fit_prompt(pairs, test_input, max_prompt)\\n    kp = [pairs[i] for i in keep]\\n    augs = G.make_augs(max(n_gen, n_score), len(kp), seed)\\n    prompts = []\\n    for a in augs:\\n        ap, at = a.apply_task(kp, [test_input])\\n        prompts.append(F.prompt_ids(ap, at[0]))\\n    cands = {}\\n    gen_info = []\\n    bs = max(1, BATCH_TOKENS // (max(len(p) for p in prompts) + F.MAX_OUT_TOKENS))\\n    i = 0\\n    while i < n_gen:\\n        idx = list(range(i, min(n_gen, i + bs)))\\n        for j, (toks, lp) in zip(idx, lm.greedy([prompts[j] for j in idx])):\\n            g = F.parse_ids(toks)\\n            if g is None or not toks or toks[-1] != F.IME:\\n                gen_info.append({\\"aug\\": j, \\"valid\\": False})\\n                continue\\n            orig = augs[j].inv(np.asarray(g))\\n            k = G.key(orig)\\n            c = cands.setdefault(k, {\\"grid\\": G.to_list(orig), \\"votes\\": 0, \\"gen_augs\\": [], \\"gen_logp\\": []})\\n            c[\\"votes\\"] += 1\\n            c[\\"gen_augs\\"].append(j)\\n            c[\\"gen_logp\\"].append(round(lp, 4))\\n            gen_info.append({\\"aug\\": j, \\"valid\\": True})\\n        i = idx[-1] + 1\\n        if deadline is not None and time.time() > deadline:\\n            break\\n    pool = list(cands.values())\\n    t_gen = time.time() - t0\\n    n_scored = 0\\n    for c in pool:\\n        c[\\"scores\\"] = []\\n    def output():\\n        return {\\"candidates\\": pool, \\"kept_demos\\": keep, \\"prompt_tokens\\": n_prompt, \\"n_gen\\": n_gen,\\n                \\"n_score_done\\": n_scored, \\"augs\\": [a.describe() for a in augs[:max(n_gen, n_score)]],\\n                \\"gen_info\\": gen_info, \\"gen_s\\": round(t_gen, 2), \\"total_s\\": round(time.time() - t0, 2)}\\n    try:\\n        if pool:\\n            for j, a in enumerate(augs[:n_score]):\\n                if deadline is not None and time.time() > deadline and j >= 1:\\n                    break\\n                targets = [F.target_ids(a.fwd(np.asarray(c[\\"grid\\"]))) for c in pool]\\n                raw = list(lm.score(prompts[j], targets))\\n                if len(raw) != len(pool):\\n                    raise ValueError(\\"incomplete all-candidate scoring pass\\")\\n                if any(isinstance(s, (bool, np.bool_)) or not isinstance(s, (int, float, np.integer, np.floating))\\n                       or not math.isfinite(float(s)) for s in raw):\\n                    raise ValueError(\\"nonfinite or nonnumeric candidate score\\")\\n                values = [round(float(s), 4) for s in raw]\\n                # Commit this augmentation only after every candidate\'s score is valid.\\n                for c, s in zip(pool, values):\\n                    c[\\"scores\\"].append(s)\\n                n_scored += 1\\n    except Exception as error:\\n        # A reused exception object must never carry a stale prefix into another query.\\n        for attribute in (\\"completed_score_prefix\\", \\"completed_score_prefix_token\\", \\"recovered_query_index\\", \\"prefix_checkpoint_error\\"):\\n            if hasattr(error, attribute):\\n                delattr(error, attribute)\\n        if n_scored > 0 and all(len(c[\\"scores\\"]) == n_scored for c in pool):\\n            error.completed_score_prefix = copy.deepcopy(output())\\n            error.completed_score_prefix_token = _prefix_token\\n            error.completed_score_prefix[\\"recovery\\"] = {\\"kind\\": \\"complete_score_prefix\\",\\n                                                       \\"error\\": type(error).__name__}\\n        raise\\n    return output()\\n\\n\\ndef predict_task(lm, task_id, task, n_gen=8, n_score=8, max_prompt=MAX_PROMPT, deadline=None, on_query=None):\\n    pairs = [(np.asarray(p[\\"input\\"]), np.asarray(p[\\"output\\"])) for p in task[\\"train\\"]]\\n    outs = []\\n    for idx, t in enumerate(task[\\"test\\"]):\\n        prefix_token = object()\\n        try:\\n            if deadline is not None and time.time() >= deadline:\\n                raise TimeoutError(\\"task deadline reached before query\\")\\n            outs.append(predict_output(lm, pairs, np.asarray(t[\\"input\\"]), n_gen, n_score, task_seed(task_id, idx),\\n                                       max_prompt, deadline, _prefix_token=prefix_token))\\n            if on_query is not None:\\n                on_query(idx, outs[:])\\n        except Exception as e:\\n            recovered = (getattr(e, \\"completed_score_prefix\\", None)\\n                         if getattr(e, \\"completed_score_prefix_token\\", None) is prefix_token else None)\\n            if recovered is not None:\\n                outs.append(recovered)\\n                e.recovered_query_index = idx\\n                if on_query is not None:\\n                    try:\\n                        on_query(idx, outs[:])\\n                    except Exception as checkpoint_error:\\n                        # Preserve both the scoring fault and in-memory prefix if persistence fails.\\n                        e.prefix_checkpoint_error = type(checkpoint_error).__name__\\n            e.partial_outputs = outs\\n            raise\\n    return outs\\n", "sf/runner.py": "\\"\\"\\"Orchestrator and GPU workers: independent replicas (one per GPU) pull tasks from a shared queue, write each task\'s\\ncandidate pool atomically, and the orchestrator keeps submission.json valid at all times (fallbacks first, then\\nupdated atomically as pools arrive). The orchestrator is also the global watchdog: at its deadline it stops the\\nworkers and writes the final file.\\n\\n  python -m sf.runner orchestrate --challenges C --model M --out DIR --submission PATH [options]\\n  python -m sf.runner worker --rank R --challenges C --model M --out DIR --deadline UNIX_TS [options]\\n\\"\\"\\"\\nimport argparse\\nimport json\\nimport os\\nimport signal\\nimport subprocess\\nimport sys\\nimport time\\nimport traceback\\n\\nfrom . import select as S\\nfrom . import textfmt as F\\n\\n\\ndef atomic_json(path, obj):\\n    tmp = f\\"{path}.tmp{os.getpid()}\\"\\n    with open(tmp, \\"w\\") as f:\\n        json.dump(obj, f)\\n        f.flush()\\n        os.fsync(f.fileno())\\n    os.replace(tmp, path)\\n\\n\\ndef task_cost(t):\\n    n = sum(len(F.grid_ids(g)) for p in t[\\"train\\"] for g in (p[\\"input\\"], p[\\"output\\"]))\\n    return len(t[\\"test\\"]) * (n + max(len(F.grid_ids(x[\\"input\\"])) for x in t[\\"test\\"]))\\n\\n\\ndef order_tasks(ch):\\n    \\"\\"\\"Largest first, so that the long tasks do not finish last.\\"\\"\\"\\n    return sorted(ch, key=lambda k: -task_cost(ch[k]))\\n\\n\\n# ----- worker ----------------------------------------------------------------------------------------------------\\ndef claim(out, tid, rank):\\n    try:\\n        fd = os.open(os.path.join(out, \\"claims\\", tid), os.O_CREAT | os.O_EXCL | os.O_WRONLY)\\n    except FileExistsError:\\n        return False\\n    os.write(fd, str(rank).encode())\\n    os.close(fd)\\n    return True\\n\\n\\ndef peak_gb():\\n    import torch\\n    return round(torch.cuda.max_memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0\\n\\n\\ndef preserve_query_results(current, incoming):\\n    \\"\\"\\"Retain the latest nonempty pool at each original query index.\\n\\n    Do not mix scores across budgets or compact empty slots. A nonempty retry\\n    keeps the baseline choice; an empty/missing retry cannot erase a prior pool.\\n    \\"\\"\\"\\n    if not isinstance(incoming, list):\\n        return current\\n    if current is None:\\n        return incoming[:]\\n    kept = current[:]\\n    for index, result in enumerate(incoming):\\n        if index >= len(kept):\\n            kept.append(result)\\n        elif result.get(\\"candidates\\"):\\n            kept[index] = result\\n    return kept\\n\\n\\ndef selected_snapshot(task, outputs, method):\\n    selected = []\\n    for j, query in enumerate(task[\\"test\\"]):\\n        pool = outputs[j].get(\\"candidates\\", []) if outputs and j < len(outputs) else []\\n        selected.append(S.two_attempts(pool, query[\\"input\\"], S.rank(pool, method)) if pool else None)\\n    return selected\\n\\n\\ndef checkpoint_outputs(path, task_id):\\n    try:\\n        if os.path.getsize(path) > 64 * 1024 * 1024:\\n            return None\\n        with open(path) as f:\\n            value = json.load(f)\\n        if not isinstance(value, dict):\\n            return None\\n        outputs = value.get(\\"outputs\\")\\n        if (value.get(\\"task\\") == task_id and isinstance(outputs, list)\\n                and all(isinstance(o, dict) and isinstance(o.get(\\"candidates\\"), list) for o in outputs)):\\n            return outputs\\n    except (OSError, ValueError, RecursionError):\\n        pass\\n    return None\\n\\n\\ndef worker(a):\\n    import torch\\n    from .lm import LM\\n    from .predict import predict_task, task_seed\\n    ch = json.load(open(a.challenges))\\n    queue = json.load(open(os.path.join(a.out, \\"queue.json\\")))\\n    log = open(os.path.join(a.out, \\"workers\\", f\\"{a.rank}.log\\"), \\"a\\")\\n\\n    def say(**kw):\\n        kw[\\"t\\"] = round(time.time(), 1)\\n        log.write(json.dumps(kw) + \\"\\\\n\\")\\n        log.flush()\\n    dev = os.environ.get(\\"SF_DEVICE\\", \\"cuda:0\\")\\n    lm = LM(a.model, device=dev)\\n    say(event=\\"loaded\\", load_s=round(lm.load_s, 1), dtype=str(lm.dtype),\\n        gpu=torch.cuda.get_device_name(0) if dev.startswith(\\"cuda\\") else dev)\\n    ttt = None\\n    if a.ttt_steps > 0:\\n        from .ttt import TTT\\n        ttt = TTT(lm, rank=a.ttt_rank, alpha=a.ttt_alpha, lr=a.ttt_lr, max_tokens=a.ttt_max_tokens)\\n        import peft\\n        say(event=\\"ttt\\", rank=a.ttt_rank, alpha=a.ttt_alpha, lr=a.ttt_lr, steps=a.ttt_steps, params=ttt.n_params,\\n            peft=peft.__version__, peft_file=peft.__file__)\\n    done = 0\\n    for tid in queue:\\n        if time.time() > a.deadline - 30:\\n            say(event=\\"deadline\\")\\n            break\\n        pool_path = os.path.join(a.out, \\"pools\\", f\\"{tid}.json\\")\\n        if os.path.exists(pool_path) or not claim(a.out, tid, a.rank):\\n            continue\\n        left = sum(1 for k in queue if not os.path.exists(os.path.join(a.out, \\"claims\\", k)))\\n        share = (a.deadline - time.time() - a.reserve_s) / max(1.0, left / a.n_workers + 1)\\n        task_deadline = min(a.deadline, time.time() + max(20.0, min(a.task_cap_s, share)))\\n        t0 = time.time()\\n        progress_path = os.path.join(a.out, \\"progress\\", f\\"{tid}.json\\")\\n        res, err, tstats = checkpoint_outputs(progress_path, tid), None, None\\n\\n        def save_query(index, outputs):\\n            nonlocal res\\n            res = preserve_query_results(res, outputs)\\n            try:\\n                atomic_json(progress_path, {\\"task\\": tid, \\"outputs\\": res,\\n                            \\"selected\\": selected_snapshot(ch[tid], res, a.method),\\n                            \\"query\\": index, \\"rank\\": a.rank, \\"stage\\": \\"QUERY_CHECKPOINT\\"})\\n            except OSError as error:\\n                say(event=\\"checkpoint_error\\", error=type(error).__name__)\\n\\n        if ttt is not None:                   # a fresh adapter per task, trained on its demonstrations only\\n            import numpy as np\\n            seed = task_seed(tid, 99)\\n            pairs = [(np.asarray(p[\\"input\\"]), np.asarray(p[\\"output\\"])) for p in ch[tid][\\"train\\"]]\\n            try:\\n                tstats = ttt.train(pairs, a.ttt_steps, seed, deadline=t0 + a.ttt_frac * (task_deadline - t0))\\n            except Exception as e:\\n                torch.cuda.empty_cache()\\n                ttt.reset(seed)                 # back to the base model for this task\\n                tstats = {\\"error\\": type(e).__name__}\\n        for attempt in range(2):\\n            try:\\n                kw = dict(n_gen=a.n_gen, n_score=a.n_score, max_prompt=a.max_prompt, deadline=task_deadline, on_query=save_query)\\n                if attempt:\\n                    kw.update(n_gen=max(1, a.n_gen // 2), n_score=max(1, a.n_score // 2),\\n                              max_prompt=a.max_prompt // 2)\\n                incoming = predict_task(lm, tid, ch[tid], **kw)\\n                res = preserve_query_results(res, incoming)\\n                break\\n            except torch.cuda.OutOfMemoryError as e:\\n                res = preserve_query_results(res, getattr(e, \\"partial_outputs\\", None))\\n                err = \\"oom\\"\\n                torch.cuda.empty_cache()\\n            except Exception as e:\\n                res = preserve_query_results(res, getattr(e, \\"partial_outputs\\", None))\\n                err = traceback.format_exc(limit=3)\\n                torch.cuda.empty_cache()\\n        atomic_json(pool_path, {\\"task\\": tid, \\"outputs\\": res, \\"selected\\": selected_snapshot(ch[tid], res, a.method), \\"error\\": None if res is not None and len(res) == len(ch[tid][\\"test\\"]) else err,\\n                                \\"rank\\": a.rank, \\"wall_s\\": round(time.time() - t0, 2), \\"ttt\\": tstats,\\n                                \\"peak_gb\\": peak_gb(), \\"static_cache\\": lm.use_static})\\n        done += 1\\n        say(event=\\"task\\", task=tid, wall_s=round(time.time() - t0, 2), ok=res is not None and len(res) == len(ch[tid][\\"test\\"]), err=err,\\n            partial=res is not None and len(res) < len(ch[tid][\\"test\\"]),\\n            n_cands=[len(o[\\"candidates\\"]) for o in res] if res else None)\\n    say(event=\\"exit\\", done=done, stats=lm.stats, peak_gb=peak_gb(), static_cache=lm.use_static)\\n\\n\\n# ----- orchestrator ----------------------------------------------------------------------------------------------\\ndef build_submission(ch, out, method, selector=None):\\n    sub, n_pool = {}, 0\\n    for tid, t in ch.items():\\n        p = os.path.join(out, \\"pools\\", f\\"{tid}.json\\")\\n        res = None\\n        if os.path.exists(p):\\n            try:\\n                res = json.load(open(p)).get(\\"outputs\\")\\n            except (OSError, ValueError):\\n                res = None\\n        entry = []\\n        for j, x in enumerate(t[\\"test\\"]):\\n            pool = res[j][\\"candidates\\"] if res and j < len(res) else []\\n            if pool:\\n                n_pool += 1\\n            order = selector(tid, j, pool) if selector else S.rank(pool, method)\\n            entry.append(S.two_attempts(pool, x[\\"input\\"], order))\\n        sub[tid] = entry\\n    return sub, n_pool\\n\\n\\ndef release_failed_worker_claims(out, queue, rank):\\n    \\"\\"\\"Release only unfinished claims owned by a worker already verified dead.\\n\\n    Called by the sole orchestrator after poll() reports a nonzero exit, before\\n    its replacement starts. Completed pools and other workers\' claims stay put.\\n    \\"\\"\\"\\n    released = []\\n    for tid in queue:\\n        if os.path.exists(os.path.join(out, \\"pools\\", f\\"{tid}.json\\")):\\n            continue\\n        path = os.path.join(out, \\"claims\\", tid)\\n        try:\\n            with open(path) as f:\\n                owner = f.read()\\n            if owner == str(rank):\\n                os.unlink(path)\\n                released.append(tid)\\n        except FileNotFoundError:\\n            pass\\n    return released\\n\\n\\ndef orchestrate(a):\\n    t_start = time.time()\\n    ch = json.load(open(a.challenges))\\n    if a.task_ids:\\n        ids = [x for x in a.task_ids.split(\\",\\") if x in ch]\\n        ch = {k: ch[k] for k in ids}\\n    elif a.limit:\\n        ch = {k: ch[k] for k in list(ch)[:a.limit]}\\n    for d in (\\"claims\\", \\"pools\\", \\"workers\\", \\"progress\\"):\\n        os.makedirs(os.path.join(a.out, d), exist_ok=True)\\n    sub_ch = os.path.join(a.out, \\"challenges_used.json\\")\\n    atomic_json(sub_ch, ch)\\n    atomic_json(os.path.join(a.out, \\"queue.json\\"), order_tasks(ch))\\n    sub, _ = build_submission(ch, a.out, a.method)\\n    atomic_json(a.submission, sub)                         # valid fallback file before any GPU work\\n    deadline = t_start + a.deadline_s\\n    worker_deadline = deadline - min(a.final_reserve_s, 0.1 * a.deadline_s)\\n    procs = {}\\n\\n    def spawn(r):\\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(r), PYTHONUNBUFFERED=\\"1\\")\\n        cmd = [sys.executable, \\"-m\\", \\"sf.runner\\", \\"worker\\", \\"--rank\\", str(r), \\"--challenges\\", sub_ch,\\n               \\"--model\\", a.model, \\"--out\\", a.out, \\"--deadline\\", str(worker_deadline), \\"--n-gen\\", str(a.n_gen),\\n               \\"--n-score\\", str(a.n_score), \\"--max-prompt\\", str(a.max_prompt), \\"--task-cap-s\\", str(a.task_cap_s),\\n               \\"--reserve-s\\", str(a.reserve_s), \\"--n-workers\\", str(a.gpus), \\"--method\\", a.method, \\"--ttt-steps\\", str(a.ttt_steps),\\n               \\"--ttt-rank\\", str(a.ttt_rank), \\"--ttt-alpha\\", str(a.ttt_alpha), \\"--ttt-lr\\", str(a.ttt_lr),\\n               \\"--ttt-max-tokens\\", str(a.ttt_max_tokens), \\"--ttt-frac\\", str(a.ttt_frac)]\\n        errf = open(os.path.join(a.out, \\"workers\\", f\\"{r}.stderr\\"), \\"a\\")\\n        return subprocess.Popen(cmd, env=env, stdout=errf, stderr=errf, cwd=a.code_dir)\\n    for r in range(a.gpus):\\n        procs[r] = spawn(r)\\n    restarts = {r: 0 for r in procs}\\n    status = {\\"start\\": t_start, \\"deadline\\": deadline}\\n    while True:\\n        time.sleep(a.poll_s)\\n        n_done = sum(1 for f in os.listdir(os.path.join(a.out, \\"pools\\")) if f.endswith(\\".json\\"))\\n        sub, n_pool = build_submission(ch, a.out, a.method)\\n        atomic_json(a.submission, sub)\\n        alive = {r: p.poll() is None for r, p in procs.items()}\\n        for r, ok in alive.items():\\n            if not ok and procs[r].returncode != 0 and restarts[r] < 2 and time.time() < worker_deadline - 120:\\n                release_failed_worker_claims(a.out, order_tasks(ch), r)\\n                restarts[r] += 1\\n                procs[r] = spawn(r)\\n                alive[r] = True\\n        status.update(t=time.time(), pools=n_done, outputs_with_pool=n_pool, alive=alive, restarts=restarts)\\n        atomic_json(os.path.join(a.out, \\"status.json\\"), status)\\n        if n_done >= len(ch) or not any(alive.values()):\\n            break\\n        if time.time() > deadline:\\n            status[\\"watchdog\\"] = \\"deadline reached; workers stopped\\"\\n            break\\n    for p in procs.values():\\n        if p.poll() is None:\\n            p.send_signal(signal.SIGTERM)\\n    for p in procs.values():\\n        try:\\n            p.wait(timeout=30)\\n        except subprocess.TimeoutExpired:\\n            p.kill()\\n    sub, n_pool = build_submission(ch, a.out, a.method)\\n    atomic_json(a.submission, sub)\\n    problems = S.check_submission(sub, ch)\\n    status.update(t=time.time(), wall_s=round(time.time() - t_start, 1), final_outputs_with_pool=n_pool,\\n                  schema_problems=problems[:20], n_schema_problems=len(problems))\\n    atomic_json(os.path.join(a.out, \\"status.json\\"), status)\\n    print(json.dumps({k: status[k] for k in (\\"wall_s\\", \\"final_outputs_with_pool\\", \\"n_schema_problems\\")}))\\n    return status\\n\\n\\ndef main(argv=None):\\n    ap = argparse.ArgumentParser()\\n    ap.add_argument(\\"mode\\", choices=[\\"orchestrate\\", \\"worker\\"])\\n    ap.add_argument(\\"--challenges\\", required=True)\\n    ap.add_argument(\\"--model\\", required=True)\\n    ap.add_argument(\\"--out\\", required=True)\\n    ap.add_argument(\\"--submission\\")\\n    ap.add_argument(\\"--gpus\\", type=int, default=4)\\n    ap.add_argument(\\"--rank\\", type=int, default=0)\\n    ap.add_argument(\\"--deadline\\", type=float)\\n    ap.add_argument(\\"--deadline-s\\", type=float, default=11.5 * 3600)\\n    ap.add_argument(\\"--final-reserve-s\\", type=float, default=600)\\n    ap.add_argument(\\"--reserve-s\\", type=float, default=300)\\n    ap.add_argument(\\"--task-cap-s\\", type=float, default=900)\\n    ap.add_argument(\\"--n-gen\\", type=int, default=8)\\n    ap.add_argument(\\"--n-score\\", type=int, default=16)\\n    ap.add_argument(\\"--max-prompt\\", type=int, default=12000)\\n    ap.add_argument(\\"--n-workers\\", type=int, default=4)\\n    ap.add_argument(\\"--poll-s\\", type=float, default=20)\\n    ap.add_argument(\\"--limit\\", type=int, default=0)\\n    ap.add_argument(\\"--task-ids\\", default=\\"\\")\\n    ap.add_argument(\\"--method\\", default=\\"vote_score\\")\\n    ap.add_argument(\\"--ttt-steps\\", type=int, default=0)\\n    ap.add_argument(\\"--ttt-rank\\", type=int, default=64)\\n    ap.add_argument(\\"--ttt-alpha\\", type=int, default=64)\\n    ap.add_argument(\\"--ttt-lr\\", type=float, default=1e-4)\\n    ap.add_argument(\\"--ttt-max-tokens\\", type=int, default=8192)\\n    ap.add_argument(\\"--ttt-frac\\", type=float, default=0.7)\\n    ap.add_argument(\\"--code-dir\\", default=os.path.dirname(os.path.dirname(os.path.abspath(__file__))))\\n    a = ap.parse_args(argv)\\n    if a.mode == \\"worker\\":\\n        worker(a)\\n    else:\\n        orchestrate(a)\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "sf/select.py": "\\"\\"\\"Selection of two attempts from a fixed candidate pool, and the submission schema check.\\n\\nNeural rules (no CORA): votes = number of decoding augmentations that produced the candidate; score = mean\\nteacher-forced log-probability over the first k scoring augmentations.\\n\\"\\"\\"\\nimport numpy as np\\n\\nfrom . import grids as G\\n\\n\\ndef mean_score(c, k=None):\\n    s = c.get(\\"scores\\") or []\\n    s = s[:k] if k else s\\n    return float(np.mean(s)) if s else float(\\"-inf\\")\\n\\n\\ndef rank(pool, method=\\"vote_score\\", k=None):\\n    \\"\\"\\"Candidate indices, best first. Ties are broken by the generation order (stable).\\"\\"\\"\\n    ix = list(range(len(pool)))\\n    if method == \\"vote_score\\":\\n        key = lambda i: (-pool[i][\\"votes\\"], -mean_score(pool[i], k))\\n    elif method == \\"score\\":\\n        key = lambda i: (-mean_score(pool[i], k),)\\n    elif method == \\"vote_gen\\":\\n        key = lambda i: (-pool[i][\\"votes\\"], -max(pool[i][\\"gen_logp\\"]))\\n    elif method == \\"score_plus_logvotes\\":\\n        key = lambda i: (-(mean_score(pool[i], k) + 2.0 * np.log(pool[i][\\"votes\\"] + 1e-9)),)\\n    else:\\n        raise ValueError(method)\\n    return sorted(ix, key=key)\\n\\n\\ndef fallback_grids(test_input):\\n    g = G.to_list(np.asarray(test_input))\\n    return [g, g]\\n\\n\\ndef two_attempts(pool, test_input, order):\\n    picks = [pool[i][\\"grid\\"] for i in order[:2]]\\n    fb = fallback_grids(test_input)\\n    while len(picks) < 2:\\n        picks.append(fb[len(picks)])\\n    return {\\"attempt_1\\": picks[0], \\"attempt_2\\": picks[1]}\\n\\n\\ndef check_submission(sub, challenges):\\n    \\"\\"\\"Schema check against the challenge file: every task, one entry per test input, two valid grids each.\\"\\"\\"\\n    problems = []\\n    if not isinstance(sub, dict):\\n        return [\\"submission is not a dict\\"]\\n    for tid, t in challenges.items():\\n        e = sub.get(tid)\\n        if not isinstance(e, list) or len(e) != len(t[\\"test\\"]):\\n            problems.append(f\\"{tid}: wrong entry count\\")\\n            continue\\n        for j, a in enumerate(e):\\n            if not isinstance(a, dict) or set(a) != {\\"attempt_1\\", \\"attempt_2\\"}:\\n                problems.append(f\\"{tid}[{j}]: keys\\")\\n                continue\\n            for k in (\\"attempt_1\\", \\"attempt_2\\"):\\n                if not G.is_valid_grid(a[k]):\\n                    problems.append(f\\"{tid}[{j}].{k}: invalid grid\\")\\n    extra = set(sub) - set(challenges)\\n    if extra:\\n        problems.append(f\\"{len(extra)} unknown task ids\\")\\n    return problems\\n", "sf/textfmt.py": "\\"\\"\\"The checkpoint\'s grid text format, as found empirically (probes/format_probe.py; PROGRESS.md, 26 Oct).\\n\\nThe vocabulary has 16 tokens: digits 0-9 are ids 0-9, newline 10, \\"user\\" 11, \\"assistant\\" 12, <|endoftext|> 13,\\n<|im_start|> 14, <|im_end|> 15. A task is a multi-turn chat: each demonstration input is a user turn and its output\\nthe following assistant turn; the test input is the last user turn and the answer is generated as the assistant turn.\\nInside a turn a grid is its rows of digits joined by newlines, with no trailing newline.\\n\\"\\"\\"\\nimport numpy as np\\n\\nNL, USER, ASSIST, EOT, IMS, IME = 10, 11, 12, 13, 14, 15\\nDIGITS = tuple(range(10))\\nMAX_SIDE = 30\\nMAX_OUT_TOKENS = MAX_SIDE * (MAX_SIDE + 1)        # 30 rows of 30 digits, 29 newlines, the end token\\n\\n\\ndef grid_ids(g):\\n    out = []\\n    for r, row in enumerate(np.asarray(g).tolist()):\\n        if r:\\n            out.append(NL)\\n        out.extend(int(v) for v in row)\\n    return out\\n\\n\\ndef user_turn(g):\\n    return [IMS, USER, NL] + grid_ids(g) + [IME, NL]\\n\\n\\ndef assistant_turn(g):\\n    return [IMS, ASSIST, NL] + grid_ids(g) + [IME, NL]\\n\\n\\nANSWER_PREFIX = [IMS, ASSIST, NL]\\n\\n\\ndef prompt_ids(pairs, test_input):\\n    ids = []\\n    for i, o in pairs:\\n        ids += user_turn(i) + assistant_turn(o)\\n    return ids + user_turn(test_input) + ANSWER_PREFIX\\n\\n\\ndef target_ids(g):\\n    \\"\\"\\"The tokens the model should emit for grid g: its rows, then the end-of-turn token.\\"\\"\\"\\n    return grid_ids(g) + [IME]\\n\\n\\ndef parse_ids(ids):\\n    \\"\\"\\"Tokens of a generated answer (end token optional) to a grid, or None if they are not a valid grid.\\"\\"\\"\\n    rows, cur = [], []\\n    for t in ids:\\n        if t == IME:\\n            break\\n        if t == NL:\\n            rows.append(cur)\\n            cur = []\\n        elif 0 <= t <= 9:\\n            cur.append(int(t))\\n        else:\\n            return None\\n    rows.append(cur)\\n    if not rows or not rows[0] or len(rows) > MAX_SIDE:\\n        return None\\n    w = len(rows[0])\\n    if w > MAX_SIDE or any(len(r) != w for r in rows):\\n        return None\\n    return rows\\n\\n\\nclass GridConstraint:\\n    \\"\\"\\"Per-sequence state for constrained decoding: every emitted sequence is a rectangular grid of at most 30 x 30\\n    followed by the end token. allowed() returns the set of permitted next tokens.\\"\\"\\"\\n\\n    __slots__ = (\\"width\\", \\"row_len\\", \\"n_rows\\", \\"done\\")\\n\\n    def __init__(self):\\n        self.width, self.row_len, self.n_rows, self.done = None, 0, 0, False\\n\\n    def allowed(self):\\n        if self.done:\\n            return ()\\n        if self.width is None:\\n            if self.row_len == 0:\\n                return DIGITS\\n            if self.row_len >= MAX_SIDE:\\n                return (NL, IME)\\n            return DIGITS + (NL, IME)\\n        if self.row_len < self.width:\\n            return DIGITS\\n        if self.n_rows + 1 >= MAX_SIDE:\\n            return (IME,)\\n        return (NL, IME)\\n\\n    def step(self, t):\\n        if t == IME:\\n            self.done = True\\n        elif t == NL:\\n            if self.width is None:\\n                self.width = self.row_len\\n            self.n_rows += 1\\n            self.row_len = 0\\n        else:\\n            self.row_len += 1\\n", "sf/ttt.py": "\\"\\"\\"Per-puzzle LoRA test-time training (pilot; authorized 26 Oct for Kaggle L4x4 only).\\n\\nOwn implementation on peft (Apache-2.0). One LoRA wrapper is created around the frozen base; before each puzzle its\\nadapter weights are re-initialised (A: Kaiming-uniform, B: zero, i.e. peft\'s default fresh adapter, so the adapted\\nmodel starts exactly at the base) with a puzzle-specific seed, and a new optimiser is built. Training uses only\\nthat puzzle\'s visible demonstrations: each sequence is a leave-one-demonstration-out chat (the other\\ndemonstrations, then the held-out one), under one pair-consistent augmentation (a D8 symmetry and a colour\\npermutation keeping 0 applied to every grid of the sequence, and a shuffled demonstration order). The loss is the\\ncross-entropy of every demonstration output after the first (each is predicted from the pairs before it).\\nTest inputs and their answers are never used.\\n\\"\\"\\"\\nimport importlib\\nimport math\\nimport random\\nimport time\\n\\nimport numpy as np\\nimport torch\\nfrom peft import LoraConfig, get_peft_model\\n\\nfrom . import grids as G\\nfrom . import textfmt as F\\n\\nTARGETS = [\\"q_proj\\", \\"k_proj\\", \\"v_proj\\", \\"o_proj\\", \\"gate_proj\\", \\"up_proj\\", \\"down_proj\\"]\\n\\n\\ndef _disable_torchao():\\n    \\"\\"\\"peft probes for torchao when it dispatches LoRA layers and raises on the old torchao of the Kaggle image;\\n    torchao is not used here (no quantisation), so the probe is answered False.\\"\\"\\"\\n    for mod in (\\"peft.import_utils\\", \\"peft.tuners.lora.torchao\\", \\"peft.utils.quantization_utils\\"):\\n        try:\\n            m = importlib.import_module(mod)\\n        except Exception:\\n            continue\\n        if hasattr(m, \\"is_torchao_available\\"):\\n            m.is_torchao_available = lambda: False\\n\\n\\ndef lora_modules(model):\\n    for m in model.modules():\\n        if hasattr(m, \\"lora_A\\") and \\"default\\" in getattr(m, \\"lora_A\\", {}):\\n            yield m\\n\\n\\nclass TTT:\\n    def __init__(self, lm, rank=64, alpha=64, lr=1e-4, max_tokens=8192, grad_ckpt=True, targets=None):\\n        self.lm, self.rank, self.alpha, self.lr, self.max_tokens = lm, rank, alpha, lr, max_tokens\\n        _disable_torchao()\\n        cfg = LoraConfig(r=rank, lora_alpha=alpha, lora_dropout=0.0, bias=\\"none\\", task_type=\\"CAUSAL_LM\\",\\n                         target_modules=targets or TARGETS)\\n        self.model = get_peft_model(lm.model, cfg)\\n        lm.model = self.model                              # inference now runs through the adapter\\n        if grad_ckpt:\\n            self.model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\\"use_reentrant\\": False})\\n        self.params = [p for p in self.model.parameters() if p.requires_grad]\\n        self.n_params = sum(p.numel() for p in self.params)\\n        self.model.eval()\\n\\n    def reset(self, seed):\\n        \\"\\"\\"A fresh adapter: the adapted model equals the frozen base again.\\"\\"\\"\\n        g = torch.Generator().manual_seed(seed)\\n        for p in self.params:                              # no gradient survives an interrupted step\\n            p.grad = None\\n        with torch.no_grad():\\n            for m in lora_modules(self.model):\\n                a, b = m.lora_A[\\"default\\"].weight, m.lora_B[\\"default\\"].weight\\n                bound = 1.0 / math.sqrt(a.shape[1])        # kaiming_uniform_(a=sqrt(5)) bound for fan_in\\n                a.copy_(torch.empty(a.shape).uniform_(-bound, bound, generator=g).to(a.dtype))\\n                b.zero_()\\n\\n    def examples(self, pairs, seed):\\n        \\"\\"\\"An endless deterministic stream of (ids, labels) training sequences from the demonstrations.\\"\\"\\"\\n        rng = random.Random(seed)\\n        n = len(pairs)\\n        k = 0\\n        while True:\\n            hold = k % n\\n            k += 1\\n            t = rng.randrange(8)\\n            perm = G.random_perm(rng, keep_zero=True)\\n            aug = G.Aug(t, perm)\\n            ctx = [j for j in range(n) if j != hold]\\n            rng.shuffle(ctx)\\n            order = ctx + [hold]\\n            seq = [(aug.fwd(pairs[j][0]), aug.fwd(pairs[j][1])) for j in order]\\n            ids, labels = self._encode(seq)\\n            while len(ids) > self.max_tokens and len(seq) > 1:\\n                longest = max(range(len(seq) - 1), key=lambda i: len(F.grid_ids(seq[i][0])) +\\n                              len(F.grid_ids(seq[i][1])))\\n                seq.pop(longest)\\n                ids, labels = self._encode(seq)\\n            yield ids, labels\\n\\n    @staticmethod\\n    def _encode(seq):\\n        \\"\\"\\"Chat tokens of the pairs; labels on every output after the first (on the only one if it is alone).\\"\\"\\"\\n        ids, labels = [], []\\n        for idx, (i, o) in enumerate(seq):\\n            u = F.user_turn(i)\\n            ids += u\\n            labels += [-100] * len(u)\\n            head = [F.IMS, F.ASSIST, F.NL]\\n            body = F.grid_ids(o) + [F.IME]\\n            ids += head + body + [F.NL]\\n            labels += [-100] * len(head) + (body if idx > 0 or len(seq) == 1 else [-100] * len(body)) + [-100]\\n        return ids, labels\\n\\n    def train(self, pairs, steps, seed, deadline=None, log_every=10):\\n        \\"\\"\\"Adapt a fresh adapter on the demonstrations. Returns statistics (loss curve, timing, memory).\\"\\"\\"\\n        self.reset(seed)\\n        opt = torch.optim.AdamW(self.params, lr=self.lr, weight_decay=0.0, betas=(0.9, 0.999))\\n        opt.zero_grad(set_to_none=True)\\n        warm = max(1, steps // 10)\\n\\n        def lr_at(s):\\n            if s < warm:\\n                return (s + 1) / warm\\n            return max(0.0, (steps - s) / max(1, steps - warm))\\n        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)\\n        dev = self.lm.device\\n        stream = self.examples(pairs, seed)\\n        self.model.train()\\n        t0, tokens, curve, done = time.time(), 0, [], 0\\n        use_amp = self.lm.dtype == torch.bfloat16\\n        try:\\n            for s in range(steps):\\n                if deadline is not None and time.time() > deadline:\\n                    break\\n                ids, labels = next(stream)\\n                x = torch.tensor([ids], device=dev)\\n                y = torch.tensor([labels], device=dev)\\n                with torch.autocast(device_type=dev.type, dtype=torch.bfloat16, enabled=use_amp):\\n                    logits = self.model(input_ids=x, use_cache=False).logits[0, :-1].float()\\n                loss = torch.nn.functional.cross_entropy(logits, y[0, 1:], ignore_index=-100)\\n                loss.backward()\\n                torch.nn.utils.clip_grad_norm_(self.params, 1.0)\\n                opt.step()\\n                sched.step()\\n                opt.zero_grad(set_to_none=True)\\n                tokens += len(ids)\\n                done += 1\\n                if s % log_every == 0 or s == steps - 1:\\n                    curve.append((s, round(float(loss.detach()), 4)))\\n        finally:\\n            self.model.eval()\\n            del opt\\n        return {\\"steps\\": done, \\"tokens\\": tokens, \\"train_s\\": round(time.time() - t0, 2), \\"loss_curve\\": curve,\\n                \\"rank\\": self.rank, \\"alpha\\": self.alpha, \\"lr\\": self.lr}\\n"}')
for name, text in FILES.items():
    p = os.path.join(CODE_DIR, name)
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, 'w', encoding='utf-8').write(text)
print('code written', CODE_SHA, len(FILES))

open(os.path.join(CODE_DIR, 'deployment_guard.py'), 'w').write('"""Stdlib parent guard: fallback before model discovery, checkpoint salvage, wall reserve.\n\nOnly the newly spawned child session is supervised. No model, answer file,\nsignal-timer nesting, or neural selection code runs in this parent.\n"""\nfrom copy import deepcopy\nfrom dataclasses import dataclass\nimport fcntl\nimport glob\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport signal\nimport subprocess\nimport time\nimport threading\n\n\n_TERMINATION_SCOPE_ACTIVE = False\n\n\nclass TerminationRequest:\n    """Scoped cooperative SIGTERM, not an asynchronous exception or exit replay.\n\n    The previous handler is restored on exit but is not called for a signal\n    received here. No masks/preexec hooks are inherited by the child. A request\n    inside Popen waits until its return assigns the owned process before cleanup.\n    SIGINT retains the caller\'s existing behavior. Main-thread, nonnested use.\n    """\n    def __init__(self):\n        self.signum = None\n\n    @property\n    def requested(self):\n        return self.signum is not None\n\n    def _record(self, signum, frame):\n        # No locks, I/O, raising, or previous-handler calls in the signal handler.\n        if self.signum is None:\n            self.signum = int(signum)\n\n    def __enter__(self):\n        global _TERMINATION_SCOPE_ACTIVE\n        if threading.current_thread() is not threading.main_thread():\n            raise RuntimeError("guardian SIGTERM scope requires the main thread")\n        if _TERMINATION_SCOPE_ACTIVE:\n            raise RuntimeError("nested guardian SIGTERM scopes are not supported")\n        self.signum = None\n        self.previous = signal.getsignal(signal.SIGTERM)\n        signal.signal(signal.SIGTERM, self._record)\n        _TERMINATION_SCOPE_ACTIVE = True\n        return self\n\n    def __exit__(self, exc_type, exc_value, traceback):\n        global _TERMINATION_SCOPE_ACTIVE\n        try:\n            signal.signal(signal.SIGTERM, self.previous)\n        finally:\n            _TERMINATION_SCOPE_ACTIVE = False\n        return False\n\n\ndef valid_grid(g):\n    return (type(g) is list and 1 <= len(g) <= 30 and type(g[0]) is list\n            and 1 <= len(g[0]) <= 30 and all(type(r) is list and len(r) == len(g[0])\n            and all(type(v) is int and 0 <= v <= 9 for v in r) for r in g))\n\n\ndef valid_pair(pair):\n    return (type(pair) is dict and set(pair) == {"attempt_1", "attempt_2"}\n            and all(valid_grid(v) for v in pair.values()))\n\n\ndef fallback(challenges):\n    sub = {}\n    for tid, task in challenges.items():\n        if not re.fullmatch(r"[A-Za-z0-9_-]{1,128}", tid):\n            raise ValueError("unsafe task identifier")\n        if not isinstance(task.get("test"), list) or not task["test"]:\n            raise ValueError("missing test inputs")\n        sub[tid] = []\n        for item in task["test"]:\n            g = item.get("input")\n            if not valid_grid(g):\n                raise ValueError("invalid challenge input")\n            sub[tid].append({"attempt_1": deepcopy(g), "attempt_2": deepcopy(g)})\n    if not sub:\n        raise ValueError("empty challenges")\n    return sub\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + f".tmp.{os.getpid()}")\n    try:\n        with temp.open("w") as f:\n            json.dump(value, f, allow_nan=False)\n            f.flush()\n            os.fsync(f.fileno())\n        os.replace(temp, path)\n    finally:\n        temp.unlink(missing_ok=True)\n\n\ndef read_json(path):\n    try:\n        path = Path(path)\n        if path.stat().st_size > 64 * 1024 * 1024:\n            return None\n        return json.loads(path.read_text())\n    except (OSError, ValueError, RecursionError):\n        return None\n\n\ndef find_challenges(kaggle_root, rehearse=False):\n    filename = "arc-agi_evaluation_challenges.json" if rehearse else "arc-agi_test_challenges.json"\n    for directory in ("input/competitions/arc-prize-2026-arc-agi-2", "input/arc-prize-2026-arc-agi-2"):\n        path = Path(kaggle_root) / directory / filename\n        if path.is_file():\n            return path\n    hits = sorted(glob.glob(str(Path(kaggle_root) / "input" / "**" / filename), recursive=True))\n    if not hits:\n        raise FileNotFoundError("competition challenge file not found")\n    return Path(hits[0])\n\n\ndef salvage(challenges, child_submission, run_dir):\n    """Validate each query independently. Checkpoints contain already-ranked pairs.\n\n    Precedence: completed task checkpoint, in-flight query checkpoint, child\n    submission, fallback. A malformed query cannot erase valid other queries.\n    """\n    result = fallback(challenges)\n    child = read_json(child_submission)\n    if type(child) is dict:\n        for tid, entries in result.items():\n            incoming = child.get(tid)\n            if type(incoming) is list:\n                for j in range(min(len(entries), len(incoming))):\n                    if valid_pair(incoming[j]):\n                        entries[j] = incoming[j]\n    recovered = 0\n    for tid, entries in result.items():\n        for folder in ("progress", "pools"):\n            record = read_json(Path(run_dir) / folder / (tid + ".json"))\n            if type(record) is not dict or record.get("task") != tid:\n                continue\n            selected = record.get("selected")\n            if type(selected) is not list:\n                continue\n            for j in range(min(len(entries), len(selected))):\n                if valid_pair(selected[j]):\n                    entries[j] = selected[j]\n        recovered += sum(result[tid][j] != fallback({tid: challenges[tid]})[tid][j]\n                         for j in range(len(entries)))\n    return result, recovered\n\n\n@dataclass(frozen=True)\nclass Budget:\n    start: float\n    total_s: float\n    serialization_s: float\n    shutdown_s: float\n\n    def __post_init__(self):\n        if (any(not math.isfinite(x) or x < 0 for x in\n                (self.start, self.total_s, self.serialization_s, self.shutdown_s))\n                or self.total_s <= self.serialization_s + self.shutdown_s):\n            raise ValueError("invalid parent runtime budget")\n\n    def work_left(self, now=None):\n        now = time.monotonic() if now is None else now\n        return max(0.0, self.start + self.total_s - self.serialization_s - self.shutdown_s - now)\n\n    def task_seconds(self, outstanding, workers, task_cap=900, now=None):\n        # Planning helper; the outer process guard remains the global wall bound.\n        left = self.work_left(now)\n        return max(0.0, min(task_cap, left, left * workers / max(1, outstanding)))\n\n\ndef stop_owned_session(process, grace_s):\n    """Terminate only the session created by this Popen(start_new_session=True).\n\n    The session is exclusively owned for this run. No name matching or system\n    process search is used. Process groups do not include the parent/notebook.\n    """\n    try:\n        if process.poll() is None and os.getpgid(process.pid) != process.pid:\n            raise RuntimeError("owned child lost its separate process group")\n    except ProcessLookupError:\n        # The leader can exit after poll(); its descendants still occupy our\n        # separately created group and must reach the cleanup below.\n        pass\n    # A session remains ours while any of its workers survive the leader. Its\n    # identifier cannot be allocated to a new process while those members live.\n    try:\n        os.killpg(process.pid, signal.SIGTERM)\n    except ProcessLookupError:\n        process.wait(timeout=max(1, grace_s))\n        return\n    until = time.monotonic() + grace_s\n    while time.monotonic() < until:\n        process.poll()  # reap a finished leader; workers can outlive it\n        try:\n            os.killpg(process.pid, 0)\n        except ProcessLookupError:\n            return\n        time.sleep(min(.02, max(0, until - time.monotonic())))\n    try:\n        os.killpg(process.pid, signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    process.wait(timeout=max(1, grace_s))\n\n\ndef supervise(challenges, submission, run_dir, command, *, budget, env=None,\n              cwd=None, poll_s=1.0, terminate_grace_s=5.0):\n    """Prewrite fallback, supervise a fresh session, preserve checkpoints at exit.\n\n    This is a local tested safety layer, not an OS or Kaggle runtime certificate.\n    If this parent is killed, its last atomically written submission still exists.\n    """\n    submission, run_dir = Path(submission), Path(run_dir)\n    submission.parent.mkdir(parents=True, exist_ok=True)\n    lease_path = submission.parent / ".submission_guard.lock"\n    process = None\n    start = time.monotonic()\n    with TerminationRequest() as termination, lease_path.open("a+") as lease:\n        fcntl.flock(lease, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        if run_dir.exists():\n            raise FileExistsError("fresh exclusively owned run directory required")\n        run_dir.mkdir(parents=True)\n        atomic_json(submission, fallback(challenges))  # before any model/PEFT imports\n        report = {"status": "FALLBACK_WRITTEN", "child_pid": None,\n                  "run_dir": str(run_dir), "recovered_queries": 0}\n        child_env = dict(os.environ if env is None else env)\n        child_env["SF_RUN_DIR"] = str(run_dir)\n        child_env["SF_CHILD_SUBMISSION"] = str(run_dir / "child_submission.json")\n        original = float(child_env.get("SF_DEADLINE_S", "41400"))\n        child_env["SF_DEADLINE_S"] = str(min(original, budget.work_left()))\n        child_env["PYTHONDONTWRITEBYTECODE"] = "1"\n        try:\n            if termination.requested:\n                report["status"] = "PARENT_SIGTERM"\n            elif budget.work_left() <= 0:\n                report["status"] = "PREPARATION_EXHAUSTED_BUDGET"\n            else:\n                with (run_dir / "child.log").open("wb") as log:\n                    process = subprocess.Popen(command, env=child_env, cwd=cwd,\n                                               stdout=log, stderr=subprocess.STDOUT,\n                                               start_new_session=True)\n                    report.update(status="RUNNING", child_pid=process.pid)\n                    while process.poll() is None and not termination.requested:\n                        sub, recovered = salvage(challenges, submission, run_dir)\n                        atomic_json(submission, sub)\n                        report["recovered_queries"] = recovered\n                        if termination.requested:\n                            break\n                        left = budget.work_left()\n                        if left <= 0:\n                            report["status"] = "PARENT_WALL_LIMIT"\n                            break\n                        time.sleep(min(poll_s, left))\n                    if termination.requested:\n                        report["status"] = "PARENT_SIGTERM"\n                    elif report["status"] == "RUNNING":\n                        report["status"] = "CHILD_EXIT"\n        finally:\n            if process is not None:\n                try:\n                    stop_owned_session(process, min(terminate_grace_s, max(.01, budget.shutdown_s)))\n                except OSError as error:\n                    report["shutdown_error"] = type(error).__name__\n                report["child_returncode"] = process.poll()\n            if termination.requested:\n                report.update(status="PARENT_SIGTERM", termination_signal=termination.signum)\n            sub, recovered = salvage(challenges, submission, run_dir)\n            atomic_json(submission, sub)\n            report.update(recovered_queries=recovered, wall_s=time.monotonic() - start,\n                          remaining_serialization_s=budget.start + budget.total_s - time.monotonic(),\n                          schema_valid=True)\n            if termination.requested:\n                report.update(status="PARENT_SIGTERM", termination_signal=termination.signum)\n            atomic_json(run_dir / "guard_status.json", report)\n        return report\n')


In [ ]:
import json, os, sys, time, uuid
from pathlib import Path
sys.path.insert(0, CODE_DIR)
from deployment_guard import Budget, find_challenges, supervise
_root = Path(os.environ.get("SF_KAGGLE_ROOT", "/kaggle"))
_challenge = find_challenges(_root, os.environ.get("SF_REHEARSE_ON") == "evaluation")
_ch = json.loads(_challenge.read_text())
_token = uuid.uuid4().hex
_driver = _root / "working" / ("guarded_driver_" + _token + ".py")
_driver.parent.mkdir(parents=True, exist_ok=True)
_driver.write_text("CODE_DIR=" + repr(CODE_DIR) + "\nCODE_SHA=" + repr(CODE_SHA) + "\n" + '\nimport glob, json, os, subprocess, sys, time\nKROOT = os.environ.get("SF_KAGGLE_ROOT", "/kaggle")   # a local dress rehearsal sets another root\nT0 = time.time()\nMODE = os.environ.get("SF_MODE", "submit")\ndef find_model():\n    for cfg in sorted(glob.glob(KROOT + "/input/**/config.json", recursive=True)):\n        try:\n            c = json.load(open(cfg))\n        except Exception:\n            continue\n        if c.get("model_type") == "qwen3" and c.get("vocab_size") == 16:\n            return os.path.dirname(cfg)\n    raise SystemExit("checkpoint not found under /kaggle/input")\ndef find_comp():\n    for d in (KROOT + "/input/competitions/arc-prize-2026-arc-agi-2", KROOT + "/input/arc-prize-2026-arc-agi-2"):\n        if os.path.isdir(d):\n            return d\n    hits = glob.glob(KROOT + "/input/**/arc-agi_test_challenges.json", recursive=True)\n    return os.path.dirname(hits[0])\nMODEL_DIR, COMP_DIR = find_model(), find_comp()\nOUT = os.environ.get("SF_RUN_DIR", KROOT + "/working/run")\nos.makedirs(OUT, exist_ok=True)\nSUB = os.environ.get("SF_CHILD_SUBMISSION", KROOT + "/working/submission.json")\nargs = ["--model", MODEL_DIR, "--out", OUT, "--submission", SUB, "--gpus", os.environ.get("SF_GPUS", "4"),\n        "--deadline-s", os.environ.get("SF_DEADLINE_S", str(11.5 * 3600)),\n        "--n-gen", os.environ.get("SF_N_GEN", "8"), "--n-score", os.environ.get("SF_N_SCORE", "16"),\n        "--task-cap-s", os.environ.get("SF_TASK_CAP_S", "900"), "--code-dir", CODE_DIR,\n        "--ttt-steps", os.environ.get("SF_TTT_STEPS", "0"), "--ttt-rank", os.environ.get("SF_TTT_RANK", "64"),\n        "--ttt-alpha", os.environ.get("SF_TTT_ALPHA", "64"), "--ttt-lr", os.environ.get("SF_TTT_LR", "1e-4"),\n        "--ttt-max-tokens", os.environ.get("SF_TTT_MAX_TOKENS", "8192"),\n        "--ttt-frac", os.environ.get("SF_TTT_FRAC", "0.7"), "--method", os.environ.get("SF_METHOD", "vote_score")]\nif int(os.environ.get("SF_TTT_STEPS", "0")) > 0:\n    whl = [p for p in glob.glob(KROOT + "/input/**/peft-0.21.2-py3-none-any.whl", recursive=True) if os.path.isfile(p)]\n    if whl:\n        os.environ["PYTHONPATH"] = whl[0] + os.pathsep + os.environ.get("PYTHONPATH", "")\ninfo = {"mode": MODE, "model_dir": MODEL_DIR, "comp_dir": COMP_DIR, "code_sha256": CODE_SHA,\n        "rerun_env": os.environ.get("KAGGLE_IS_COMPETITION_RERUN"), "python": sys.version.split()[0]}\nfor mod in ("torch", "transformers", "accelerate", "peft", "numpy", "scipy"):\n    try:\n        info[mod] = __import__(mod).__version__\n    except Exception as e:\n        info[mod] = None\ninfo["model_files"] = {f: os.path.getsize(os.path.join(MODEL_DIR, f)) for f in sorted(os.listdir(MODEL_DIR))}\nif MODE == "dev":\n    ch_path = os.path.join(COMP_DIR, "arc-agi_evaluation_challenges.json")\n    if os.environ.get("SF_TASK_IDS"):\n        args += ["--task-ids", os.environ["SF_TASK_IDS"]]\n    elif os.environ.get("SF_LIMIT"):\n        args += ["--limit", os.environ["SF_LIMIT"]]\nelif os.environ.get("SF_REHEARSE_ON") == "evaluation":\n    # timed rehearsal: the submit path end to end, on the public evaluation challenges in place of the test file\n    ch_path = os.path.join(COMP_DIR, "arc-agi_evaluation_challenges.json")\n    info["placeholder_test_file"] = False\n    info["rehearsal"] = "public evaluation challenges (CONTAMINATED for this checkpoint)"\nelse:\n    ch_path = os.path.join(COMP_DIR, "arc-agi_test_challenges.json")\n    with open(ch_path) as f:\n        test_ids = set(json.load(f))\n    try:                                    # a missing or unreadable training file means "not the placeholder"\n        with open(os.path.join(COMP_DIR, "arc-agi_training_challenges.json")) as f:\n            train_ids = set(json.load(f))\n    except (OSError, ValueError):\n        train_ids = set()\n    placeholder = bool(test_ids) and test_ids <= train_ids\n    info["placeholder_test_file"] = placeholder\n    if placeholder:\n        n = os.environ.get("SF_COMMIT_TASKS", "8")\n        ch_full = json.load(open(ch_path))\n        keep = sorted(ch_full)[:int(n)]\n        args += ["--task-ids", ",".join(keep)]\ninfo["challenges"] = ch_path\nwith open(KROOT + "/working/run_info.json", "w") as f:\n    json.dump(info, f, indent=1)\nprint(json.dumps(info), flush=True)\ntry:\n    subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"])\nexcept OSError:\n    pass\nr = subprocess.run([sys.executable, "-m", "sf.runner", "orchestrate", "--challenges", ch_path] + args, cwd=CODE_DIR)\nif MODE != "dev":\n    # the runner wrote a file for the tasks it ran; complete it for every task of the official file\n    sys.path.insert(0, CODE_DIR)\n    from sf import select as S\n    ch_all = json.load(open(ch_path))\n    try:\n        sub = json.load(open(SUB))\n    except Exception:\n        sub = {}\n    for tid, t in ch_all.items():\n        if tid not in sub or len(sub[tid]) != len(t["test"]):\n            sub[tid] = [S.two_attempts([], x["input"], []) for x in t["test"]]\n    tmp = SUB + ".tmp"\n    with open(tmp, "w") as f:\n        json.dump(sub, f)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, SUB)\n    problems = S.check_submission(sub, ch_all)\n    print("schema problems:", len(problems), problems[:5])\nprint("returncode", r.returncode, "wall_s", round(time.time() - T0, 1))\n')
_run = _root / "working" / ("guarded_run_" + _token)
_budget = Budget(_GUARD_STARTED, float(os.environ.get("SF_GUARD_TOTAL_S", "42600")),
                 float(os.environ.get("SF_GUARD_SERIALIZE_S", "300")),
                 float(os.environ.get("SF_GUARD_SHUTDOWN_S", "30")))
_report = supervise(_ch, _root / "working/submission.json", _run,
                    [sys.executable, str(_driver)], budget=_budget, cwd=CODE_DIR)
print(json.dumps(_report), flush=True)
